# Week 10 Example Solution: Beat the Baseline

**Challenge:** Build a neural network (MLP) to classify left vs right motor imagery from EEG brain signals. Compare it to a logistic regression baseline.

**Dataset:** Pre-extracted EEG features from 109 participants (PhysioNet EEGBCI)

**Key finding:** The neural network did NOT beat logistic regression on this data. The added complexity was not justified.

---

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

# PyTorch — new this week!
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

IMG_DIR = Path('images')
IMG_DIR.mkdir(exist_ok=True)

print(f'PyTorch version: {torch.__version__}')
print('All imports successful!')

---

## Part 1: Load and Explore the Data

In [ ]:
# Cell 2: Load the data
data = pd.read_csv('../data/eeg_motor_imagery_features.csv')

print(f'Dataset: {data.shape[0]:,} trials × {data.shape[1]} columns')
print(f'Participants: {data["participant_id"].nunique()}')
print()
data.head()

In [ ]:
# Cell 3: Class balance
print('Class balance:')
print(data['condition'].value_counts())
print()

fig, ax = plt.subplots(figsize=(6, 4))
colours = ['#4A90D9', '#A71930']
data['condition'].value_counts().plot(kind='bar', color=colours, ax=ax, edgecolor='white')
ax.set_title('Class Balance: Left vs Right Motor Imagery', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of trials')
ax.set_xticklabels(['Left hand', 'Right hand'], rotation=0)

for i, (label, count) in enumerate(data['condition'].value_counts().items()):
    ax.text(i, count + 30, f'{count:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(IMG_DIR / 'class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4: Feature overview
feature_cols = [c for c in data.columns
                if c not in ['participant_id', 'trial', 'condition', 'condition_code']]

print(f'Number of features: {len(feature_cols)}')
print(f'Feature value range: {data[feature_cols].min().min():.4f} to {data[feature_cols].max().max():.4f}')
print()

# Check motor cortex channels (C3 and C4)
for ch in ['C3', 'C4']:
    print(f'{ch} band powers (mean by condition):')
    for band in ['alpha', 'beta']:
        col = f'{ch}_{band}'
        if col in data.columns:
            left_mean = data[data['condition'] == 'left'][col].mean()
            right_mean = data[data['condition'] == 'right'][col].mean()
            print(f'  {band:>5}: Left={left_mean:.4f}, Right={right_mean:.4f}, Diff={right_mean - left_mean:+.4f}')
    print()

In [ ]:
# Cell 5: Motor cortex feature distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
motor_cols = ['C3_alpha', 'C4_alpha', 'C3_beta', 'C4_beta']
titles = ['C3 Alpha (left motor cortex)', 'C4 Alpha (right motor cortex)',
          'C3 Beta (left motor cortex)', 'C4 Beta (right motor cortex)']

for ax, col, title in zip(axes.flat, motor_cols, titles):
    if col in data.columns:
        for condition, colour in zip(['left', 'right'], colours):
            subset = data[data['condition'] == condition][col]
            ax.hist(subset, bins=40, alpha=0.6, color=colour, label=f'{condition.title()} hand',
                    edgecolor='white', linewidth=0.5)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Band power (µV²/Hz)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=9)

plt.suptitle('Motor Cortex Feature Distributions by Condition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(IMG_DIR / 'motor_cortex_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('The distributions overlap heavily — the signal is weak. 60% accuracy is expected.')

---

## Part 2: Prepare the Data

In [ ]:
# Cell 6: Split and standardise
X = data[feature_cols].values
y = data['condition_code'].values

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} trials')
print(f'Test:  {len(X_test):,} trials')

# Standardise features (CRITICAL for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use train stats only!

print(f'\nFeatures standardised:')
print(f'  Train mean: {X_train_scaled.mean():.6f} (should be ~0)')
print(f'  Train std:  {X_train_scaled.std():.6f} (should be ~1)')

---

## Phase 1: Logistic Regression Baseline

This should feel familiar from Weeks 4 and 6. Logistic regression is our reference point — whatever accuracy it achieves, the neural network needs to beat it.

In [ ]:
# Cell 7: Logistic Regression Baseline
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

print(f'Logistic Regression:')
print(f'  Accuracy: {lr_acc:.3f}')
print(f'  F1 Score: {lr_f1:.3f}')
print()
print('Classification Report:')
print(classification_report(y_test, lr_pred, target_names=['Left', 'Right']))

print(f'\nTarget to beat: {lr_acc:.3f} accuracy')

---

## Phase 2: Build a Neural Network in PyTorch

This is new territory. We need to:
1. Convert data to PyTorch tensors
2. Define a network architecture (nn.Module)
3. Write a training loop
4. Track and plot the loss

In [ ]:
# Cell 8: Convert to PyTorch tensors and create a DataLoader
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor  = torch.FloatTensor(X_test_scaled)
y_test_tensor  = torch.LongTensor(y_test)

# Wrap the training data in a DataLoader so we can iterate in mini-batches.
# This is the inner BATCH LOOP from the Week 9 flow chart.
batch_size = 32
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True,
)

print(f'X_train tensor: {X_train_tensor.shape} (dtype: {X_train_tensor.dtype})')
print(f'y_train tensor: {y_train_tensor.shape} (dtype: {y_train_tensor.dtype})')
print(f'X_test tensor:  {X_test_tensor.shape}')
print(f'y_test tensor:  {y_test_tensor.shape}')
print(f'DataLoader:     {len(train_loader)} batches of {batch_size} '
      f'({len(train_loader.dataset)} training samples)')

In [ ]:
# Cell 9: Define the neural network
class SimpleNN(nn.Module):
    """Simple MLP: input → 64 hidden (ReLU + Dropout) → 2 output."""

    def __init__(self, n_features):
        super().__init__()
        self.layer1 = nn.Linear(n_features, 64)   # 320 inputs → 64 hidden
        self.relu = nn.ReLU()                       # activation function
        self.dropout = nn.Dropout(0.3)              # regularisation
        self.output = nn.Linear(64, 2)              # 64 hidden → 2 classes

    def forward(self, x):
        x = self.dropout(self.relu(self.layer1(x)))
        return self.output(x)


# Set the random seed BEFORE creating the model so weights initialise
# reproducibly. Without this you'd get slightly different numbers each run.
torch.manual_seed(42)

n_features = X_train_scaled.shape[1]
model = SimpleNN(n_features)

print('Model architecture:')
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 10: Training loop — nested epoch + batch loops
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 50
train_losses = []
val_losses = []

for epoch in range(n_epochs):
    # ----- outer EPOCH loop -----
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        # ----- inner BATCH loop -----
        optimizer.zero_grad()                          # reset gradients
        outputs = model(X_batch)                       # forward pass
        loss = criterion(outputs, y_batch)             # compute loss
        loss.backward()                                # backward pass
        optimizer.step()                               # update weights
        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    # After all batches: evaluate on the held-out test set (no gradients)
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_tensor)
        val_loss = criterion(val_outputs, y_test_tensor)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{n_epochs}: '
              f'Train loss={train_losses[-1]:.4f}, '
              f'Val loss={val_losses[-1]:.4f}')

print(f'\nTraining complete!')
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val loss:   {val_losses[-1]:.4f}')
print(f'Best val loss:    {min(val_losses):.4f} (epoch {np.argmin(val_losses)+1})')

---

## Training Diagnostics

The loss curve is our most important diagnostic:
- **Decreasing:** The network is learning
- **Flat at ~0.693:** The network is guessing randomly (ln(2) = 0.693)
- **Train drops, val rises:** Overfitting

In [ ]:
# Cell 11: Plot loss curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Training loss', color='#4A90D9', linewidth=2)
ax.plot(val_losses, label='Validation loss', color='#A71930', linewidth=2)
ax.axhline(y=np.log(2), color='gray', linestyle='--', linewidth=1,
           label=f'Random guessing (ln(2) = {np.log(2):.3f})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (CrossEntropy)', fontsize=12)
ax.set_title('Training and Validation Loss Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, n_epochs)
plt.tight_layout()
plt.savefig(IMG_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12: Annotated loss diagnostics
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_losses, label='Training loss', color='#4A90D9', linewidth=2)
ax.plot(val_losses, label='Validation loss', color='#A71930', linewidth=2)
ax.axhline(y=np.log(2), color='gray', linestyle='--', linewidth=1, alpha=0.7,
           label=f'Random guessing ({np.log(2):.3f})')

# Annotate key features
ax.annotate('Network starts learning\n(loss drops below chance)',
            xy=(10, train_losses[min(10, len(train_losses)-1)]),
            xytext=(25, np.log(2) + 0.01),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='gray'))

min_val_idx = np.argmin(val_losses)
if min_val_idx < n_epochs - 1:
    ax.annotate(f'Best validation loss\n(epoch {min_val_idx+1})',
                xy=(min_val_idx, val_losses[min_val_idx]),
                xytext=(min_val_idx + 15, val_losses[min_val_idx] + 0.02),
                fontsize=10, ha='center',
                arrowprops=dict(arrowstyle='->', color='#A71930', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor=(167/255, 25/255, 48/255, 0.1),
                          edgecolor='#A71930'))

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (CrossEntropy)', fontsize=12)
ax.set_title('Training Diagnostics: Loss Curve Analysis', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, n_epochs)
plt.tight_layout()
plt.savefig(IMG_DIR / 'loss_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Phase 3: Compare and Reflect

In [ ]:
# Cell 13: Evaluate MLP on test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    nn_pred = test_outputs.argmax(dim=1).numpy()

nn_acc = accuracy_score(y_test, nn_pred)
nn_f1 = f1_score(y_test, nn_pred)

print('=== Model Comparison ===')
print(f'{"Model":<25} {"Accuracy":>10} {"F1":>10}')
print('-' * 47)
print(f'{"Random guessing":<25} {"0.500":>10} {"—":>10}')
print(f'{"Logistic Regression":<25} {lr_acc:>10.3f} {lr_f1:>10.3f}')
print(f'{"Simple MLP (1 hidden)":<25} {nn_acc:>10.3f} {nn_f1:>10.3f}')
print()

diff = nn_acc - lr_acc
if diff > 0.005:
    print(f'MLP improvement: +{diff:.3f} accuracy ({diff*100:.1f} percentage points)')
elif diff < -0.005:
    print(f'MLP was WORSE by {abs(diff):.3f} accuracy ({abs(diff)*100:.1f} percentage points)')
else:
    print('MLP and LogReg performed about the same.')

In [ ]:
# Cell 14: Model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

models = ['Logistic\nRegression', 'Simple\nMLP']
model_colours = ['#4A90D9', '#A71930']

# Accuracy
axes[0].bar(models, [lr_acc, nn_acc], color=model_colours, edgecolor='white', width=0.6)
axes[0].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Chance (50%)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_ylim(0.4, 0.75)
axes[0].legend(fontsize=9)
for i, v in enumerate([lr_acc, nn_acc]):
    axes[0].text(i, v + 0.008, f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

# F1
axes[1].bar(models, [lr_f1, nn_f1], color=model_colours, edgecolor='white', width=0.6)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score', fontsize=12, fontweight='bold')
axes[1].set_ylim(0.4, 0.75)
for i, v in enumerate([lr_f1, nn_f1]):
    axes[1].text(i, v + 0.008, f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

plt.suptitle('Model Comparison: LogReg vs MLP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 15: Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in zip(axes,
                           [lr_pred, nn_pred],
                           ['Logistic Regression', 'Simple MLP']):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Left', 'Right'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, pred):.3f}',
                 fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 16: Feature importance (LogReg coefficients)
coefs = np.abs(lr_model.coef_[0])
top_k = 15
top_idx = np.argsort(coefs)[-top_k:]
top_features = [feature_cols[i] for i in top_idx]
top_values = coefs[top_idx]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(top_k), top_values, color='#4A90D9', edgecolor='white')

# Highlight motor cortex channels
for i, feat in enumerate(top_features):
    if feat.startswith(('C3_', 'C4_', 'Cz_')):
        bars[i].set_color('#A71930')

ax.set_yticks(range(top_k))
ax.set_yticklabels(top_features)
ax.set_xlabel('Absolute Coefficient', fontsize=12)
ax.set_title('Top 15 Features (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#A71930', label='Motor cortex (C3/C4/Cz)'),
                   Patch(facecolor='#4A90D9', label='Other channels')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig(IMG_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Was It Worth It?

**Honest assessment:**

The neural network did NOT meaningfully beat logistic regression on this data. Both achieved roughly the same accuracy (~60%). 

**Why?** We gave both models the same pre-extracted features (band powers). The MLP had no advantage because:
- The features were already well-engineered
- The signal is genuinely weak (heavily overlapping distributions)
- 320 features with ~5,000 trials doesn't leave much room for a 20,000-parameter network to find patterns that a linear model can't

**What did we learn?**
1. More complex models don't automatically perform better
2. The loss curve is the most important diagnostic for neural networks
3. Neural networks shine when they can learn representations from raw data — not when given hand-crafted features
4. Always start with a simple baseline — it keeps you honest

**Code comparison:**
- LogReg: ~5 lines of code (fit, predict, score)
- MLP: ~30 lines (class definition, tensor conversion, training loop, evaluation)
- Plus a new framework (PyTorch) to learn

For this problem, the simple approach won.

In [ ]:
# Cell 17: Summary figure
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

summary_text = f"""
MODEL COMPARISON SUMMARY
{'='*50}

                        LogReg          MLP
  Accuracy:           {lr_acc:.3f}         {nn_acc:.3f}
  F1 Score:           {lr_f1:.3f}         {nn_f1:.3f}
  Lines of code:       ~5             ~30
  Debugging effort:    Minimal        Moderate
  Training time:       <1 sec         ~2 sec

{'='*50}
VERDICT: The neural network did NOT meaningfully beat
logistic regression on this data. The added complexity
was NOT justified.

WHY? Pre-extracted tabular features with weak signal.
Neural networks shine when they learn representations
from raw data (images, text, raw EEG waveforms).
"""

ax.text(0.5, 0.5, summary_text, transform=ax.transAxes,
        fontsize=12, fontfamily='monospace', verticalalignment='center',
        horizontalalignment='center',
        bbox=dict(boxstyle='round,pad=1', facecolor='#f8f9fa', edgecolor='#dee2e6'))

plt.tight_layout()
plt.savefig(IMG_DIR / 'summary.png', dpi=150, bbox_inches='tight')
plt.show()